# Figure 5 — β-blocker / ADRB2 / prostate cancer case study

## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

- `adrenergic_selectivity_fig5.csv`
- `Propranolol_growth.csv`, `Carvedilol_growth.csv`
- `vct/propranolol/*.csv`, `vct/carvedilol/*.csv`


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Panel a — LinkD-Select ranks for ADRB2

In [ ]:

adr = io.read_adrenergic()
adrb2 = adr[adr["Target"].astype(str).str.contains("ADRB2", case=False)].copy()
adrb2 = adrb2.sort_values("Selectivity_Score", ascending=False).reset_index(drop=True)
adrb2["rank"] = np.arange(1, len(adrb2) + 1)
# map chembl names if possible
sel = io.read_selectivity()
name_map = sel.set_index("Drug Chembl ID")["Drug Name"].to_dict() if "Drug Name" in sel.columns else {}
adrb2["Drug Name"] = adrb2["Drug"].map(name_map)
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.scatter(adrb2["rank"], adrb2["Selectivity_Score"], s=3, alpha=0.3, c="#888888")
for drug, color in [("Propranolol", "#C44E52"), ("Carvedilol", "#1ABC9C"), ("Metoprolol", "#7F8C8D")]:
    m = adrb2["Drug Name"].astype(str).str.contains(drug, case=False, na=False)
    if m.any():
        r = adrb2[m].iloc[0]
        ax.scatter([r["rank"]], [r["Selectivity_Score"]], s=40, c=color, label=f"{drug} (rank {int(r['rank'])})")
ax.legend(frameon=False)
ax.set_xlabel("Rank")
ax.set_ylabel("Selectivity_Score")
ax.set_title("Fig 5a — ADRB2 selectivity ranks")
fig.tight_layout()
out = style.save_panel(fig, "fig5_a_adrb2_ranks", adrb2[["rank", "Drug", "Drug Name", "Selectivity_Score", "Rank_Select"]])
plt.show()
print(out)


## Panels b–c — Docked poses (illustration + process)

In [ ]:
illustrate.show_panel('fig5_b', title='Panel b — propranolol / ADRB2')
illustrate.show_panel('fig5_c', title='Panel c — carvedilol / ADRB2')

## Panel d — Adrenergic receptor selectivity heatmap

In [ ]:

adr = io.read_adrenergic()
sel = io.read_selectivity()
name_map = sel.set_index("Drug Chembl ID")["Drug Name"].to_dict() if "Drug Name" in sel.columns else {}
adr["Drug Name"] = adr["Drug"].map(name_map)
focus = ["Propranolol", "Carvedilol", "Metoprolol"]
sub = adr[adr["Drug Name"].astype(str).str.contains("|".join(focus), case=False, na=False)].copy()
# normalize drug label
def canon(n):
    n = str(n).lower()
    for f in focus:
        if f.lower() in n:
            return f
    return n
sub["drug_label"] = sub["Drug Name"].map(canon)
sub["receptor"] = sub["Target"].astype(str).str.replace("_HUMAN", "", regex=False)
piv = sub.pivot_table(index="drug_label", columns="receptor", values="Selectivity_Score", aggfunc="mean")
piv = piv.reindex([f for f in focus if f in piv.index])
fig, ax = plt.subplots(figsize=(5.5, 2.2))
im = ax.imshow(piv.values, aspect="auto", cmap="magma")
ax.set_xticks(range(len(piv.columns)))
ax.set_xticklabels(piv.columns, rotation=45, ha="right")
ax.set_yticks(range(len(piv.index)))
ax.set_yticklabels(piv.index)
ax.set_title("Fig 5d — adrenergic selectivity heatmap")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
out = style.save_panel(fig, "fig5_d_heatmap", piv.reset_index())
plt.show()
print(out)
print("Note: packaged extract currently includes ADRB1/2/3; ADRA subtypes may be absent.")


## Panels e–f — LNCaP growth inhibition assays

In [ ]:

from scipy.stats import mannwhitneyu
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4), sharey=True)
for ax, drug, fname in [
    (axes[0], "Propranolol", "propranolol"),
    (axes[1], "Carvedilol", "carvedilol"),
]:
    g = io.read_growth(fname)
    g.columns = [c.strip() for c in g.columns]
    # melt
    long = g.melt(var_name="condition", value_name="viability").dropna()
    order = list(g.columns)
    data = [long.loc[long["condition"] == c, "viability"].values for c in order]
    ax.boxplot(data, labels=order, showfliers=False)
    for i, vals in enumerate(data, start=1):
        ax.scatter(np.random.normal(i, 0.05, size=len(vals)), vals, s=10, alpha=0.7, c="#333333")
    # stats vs DMSO
    if len(data) >= 2:
        for i in range(1, len(data)):
            try:
                p = mannwhitneyu(data[0], data[i], alternative="two-sided").pvalue
            except Exception:
                p = np.nan
            ax.text(i+1, max(data[i]) if len(data[i]) else 0, f"p={p:.2g}", ha="center", fontsize=6)
    ax.set_title(f"Fig 5{'e' if drug=='Propranolol' else 'f'} — {drug}")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("% viability" if drug=="Propranolol" else "")
fig.tight_layout()
# save combined source
src = pd.concat([
    io.read_growth("propranolol").assign(drug="Propranolol"),
    io.read_growth("carvedilol").assign(drug="Carvedilol"),
])
out = style.save_panel(fig, "fig5_ef_growth", src)
plt.show()
print(out)


## Panels g–k — EHR target-trial results (from VCT summary tables)
Patient-level PHI is not packaged; these panels regenerate from propensity-matched summary outputs.

In [ ]:

# Panel h/i style: cumulative incidence / HR for propranolol vs metoprolol (AvsB)
hr_p = io.read_vct("propranolol", "results_HR.csv")
hr_c = io.read_vct("carvedilol", "results_HR.csv")
inc_p = io.read_vct("propranolol", "results_incidence.csv")
# Prefer AvsB_1to1_full seed0
def pick(df):
    m = df[(df["cell"].astype(str).str.contains("AvsB_1to1_full")) & (df.get("seed", 0) == 0)].copy()
    if m.empty:
        m = df[df["cell"].astype(str).str.contains("AvsB_1to1_full")].copy()
    return m.sort_values("window_days")

hp = pick(hr_p)
hc = pick(hr_c)
ip = pick(inc_p)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
# incidence-like bars for 5y if available
ax = axes[0]
sub = ip[ip["window_label"] == "5y"]
if len(sub):
    r = sub.iloc[0]
    ax.bar(["treat", "ctrl"], [r["ir_treat"], r["ir_ctrl"]], color=["#C44E52", "#7F8C8D"])
    ax.set_title(f"Fig 5h — Prop vs Met IR 5y\nHR={hp[hp.window_label=='5y'].iloc[0]['hr']:.2f}" if len(hp[hp.window_label=='5y']) else "Fig 5h")
else:
    ax.set_title("Fig 5h — no 5y row")
ax.set_ylabel("Incidence rate")

ax = axes[1]
# carvedilol HR forest across windows
if len(hc):
    ax.errorbar(hc["hr"], np.arange(len(hc)), xerr=[hc["hr"]-hc["ci_lower"], hc["ci_upper"]-hc["hr"]], fmt="o", color="#1ABC9C")
    ax.axvline(1, color="gray", lw=0.7)
    ax.set_yticks(range(len(hc)))
    ax.set_yticklabels(hc["window_label"])
ax.set_title("Fig 5i — Carvedilol vs Met HR")
ax.set_xlabel("Hazard ratio")

ax = axes[2]
# panel j: HR across windows for both drugs
for lab, df, color in [("Propranolol", hp, "#C44E52"), ("Carvedilol", hc, "#1ABC9C")]:
    if len(df):
        ax.plot(df["window_label"], df["hr"], marker="o", label=lab, color=color)
ax.axhline(1, color="gray", lw=0.7)
ax.legend(frameon=False, fontsize=6)
ax.set_title("Fig 5j — HR by follow-up window")
ax.set_ylabel("HR")
fig.tight_layout()
out = style.save_panel(fig, "fig5_hij_ehr", pd.concat([hp.assign(drug="propranolol"), hc.assign(drug="carvedilol")], ignore_index=True))
plt.show()
print(out)


In [ ]:

# Panel k — subgroup forest for propranolol
sg = io.read_vct("propranolol", "results_subgroup.csv")
sg = sg[(sg["cell"].astype(str).str.contains("AvsB_1to1_full")) & (sg["window_label"] == "5y")].copy()
if sg.empty:
    sg = io.read_vct("propranolol", "results_subgroup.csv")
    sg = sg[sg["window_label"] == "5y"].copy()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
if len(sg):
    y = np.arange(len(sg))
    ax.errorbar(sg["hr_in_group"], y, xerr=[sg["hr_in_group"]-sg["hr_ci_lower_in_group"], sg["hr_ci_upper_in_group"]-sg["hr_in_group"]], fmt="o", color="#C44E52")
    ax.axvline(1, color="gray", lw=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels(sg["label"].astype(str), fontsize=6)
ax.set_xlabel("HR")
ax.set_title("Fig 5k — propranolol subgroup HR (5y)")
fig.tight_layout()
out = style.save_panel(fig, "fig5_k_subgroup", sg)
plt.show()
print(out)


## Panel g — PSM design (illustration from descriptive balance)

In [ ]:

desc = io.read_vct("propranolol", "descriptive_stats.csv")
desc = desc[desc["cell"].astype(str).str.contains("AvsB_1to1_full")].copy()
fig, ax = plt.subplots(figsize=(4.5, 3.5))
d = desc[desc["covariate"] != "n_matched"].dropna(subset=["smd"])
ax.axvline(0.1, color="gray", ls="--", lw=0.7)
ax.axvline(-0.1, color="gray", ls="--", lw=0.7)
ax.scatter(d["smd"], np.arange(len(d)), s=20, c="#4C72B0")
ax.set_yticks(range(len(d)))
ax.set_yticklabels(d["covariate"], fontsize=6)
ax.set_xlabel("Standardized mean difference")
ax.set_title("Fig 5g — covariate balance after PSM (SMD)")
fig.tight_layout()
out = style.save_panel(fig, "fig5_g_psm_balance", d)
plt.show()
print(out)
display(Markdown("Full PSM design narrative is in `source_data/vct/propranolol/SUMMARY.md`."))
